#**Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
movies = pd.read_csv('/content/movies (1) (1).csv')

In [3]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
27273,131254,Kein Bund für's Leben (2007),Comedy
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy
27275,131258,The Pirates (2014),Adventure
27276,131260,Rentun Ruusu (2001),(no genres listed)


In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
file = open('/content/combined_data_1.txt')

for i in range(10):
    print(file.readline())

1:

1488844,3,2005-09-06

822109,5,2005-05-13

885013,4,2005-10-19

30878,4,2005-12-26

823519,3,2004-05-03

893988,3,2005-11-17

124105,4,2004-08-05

1248029,3,2004-04-22

1842128,4,2004-05-09



In [6]:
ratings = []
movie_id = 0

with open('/content/combined_data_1.txt') as file:

    for line in file:
        line = line.strip()

        if ':' in line:
            movie_id = int(line.replace(':', ''))

        else:
            user_id, rating, date = line.split(',')
            ratings.append([user_id, movie_id, rating])

ratings_df = pd.DataFrame(ratings,columns=['userId', 'movieId', 'rating'])

In [7]:
ratings_df.head()

,userId,movieId,rating
0,1488844,1,3
1,822109,1,5
2,885013,1,4
3,30878,1,4
4,823519,1,3


In [8]:
df = pd.merge(ratings_df,movies,on='movieId')

df.head()

,userId,movieId,rating,title,genres
0,1488844,1,3,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,822109,1,5,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,885013,1,4,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,30878,1,4,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,823519,1,3,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy


In [9]:
df.shape

(23813503, 5)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23813503 entries, 0 to 23813502
Data columns (total 5 columns):
 #   Column   Dtype 
---  ------   ----- 
 0   userId   object
 1   movieId  int64 
 2   rating   object
 3   title    object
 4   genres   object
dtypes: int64(1), object(4)
memory usage: 908.4+ MB


In [11]:
df[['userId', 'rating']] = df[['userId', 'rating']].astype(int)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23813503 entries, 0 to 23813502
Data columns (total 5 columns):
 #   Column   Dtype 
---  ------   ----- 
 0   userId   int64 
 1   movieId  int64 
 2   rating   int64 
 3   title    object
 4   genres   object
dtypes: int64(3), object(2)
memory usage: 908.4+ MB


In [39]:
df

,userId,movieId,rating,title,genres
0,1488844,1,3,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,822109,1,5,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,885013,1,4,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,30878,1,4,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,823519,1,3,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
...,...,...,...,...,...
23813498,2591364,4499,2,Dirty Rotten Scoundrels (1988),Comedy
23813499,1791000,4499,2,Dirty Rotten Scoundrels (1988),Comedy
23813500,512536,4499,5,Dirty Rotten Scoundrels (1988),Comedy
23813501,988963,4499,3,Dirty Rotten Scoundrels (1988),Comedy


#**1. Find out the list of most popular and liked genre**

In [45]:
popular_genre = df.groupby('genres')['rating'].count().sort_values(ascending=False)

popular_genre.head(10)

,rating
genres,
Drama,4365603
Comedy,2485519
Horror,1161658
Comedy|Drama,1083616
Drama|Romance,1018947
Comedy|Romance,855068
Documentary,840199
Comedy|Drama|Romance,643146
Crime|Drama,548070


In [46]:
liked_genre = df.groupby('genres')['rating'].mean().sort_values(ascending=False)

liked_genre.head(10)

,rating
genres,
Action|Crime|Mystery|Thriller,4.470761
Comedy|Musical|War,4.348704
Action|Adventure|Children,4.344422
Adventure|Animation|Children|Comedy|Fantasy|Romance,4.325245
Action|Adventure|War,4.286960
Drama|Fantasy|Horror|Thriller,4.267554
Comedy|Documentary|Romance,4.254009
Action|Adventure|Mystery|Romance|Thriller,4.191339
Fantasy|Romance,4.171470


In [15]:
df.shape

(23813503, 5)

#**2. Create Model that finds the best suited Movie for one user in every genre**

In [16]:
!pip install scikit-surprise

In [24]:
sample_df = df.sample(n=100000,random_state=42)


In [25]:
from surprise import Dataset
from surprise import Reader

reader = Reader(rating_scale=(1,5))

data = Dataset.load_from_df(sample_df[['userId','movieId','rating']],reader)

In [26]:
type(data)

surprise.dataset.DatasetAutoFolds

In [27]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data,test_size=0.2,random_state=42)

In [28]:
from surprise import SVD

model = SVD()

In [29]:
model.fit(trainset)

In [30]:
predictions = model.test(testset)

In [31]:
from surprise.accuracy import rmse

rmse(predictions)

RMSE: 1.0281


np.float64(1.0280786941214473)

In [32]:
user_id = sample_df['userId'].iloc[0]

print(user_id)

2287896


In [33]:
movie_ids = sample_df['movieId'].unique()

predictions = []

for movie_id in movie_ids:
    pred = model.predict(user_id,movie_id)

    predictions.append([movie_id, pred.est])

In [34]:
recommendations = pd.DataFrame(predictions,columns=['movieId', 'predicted_rating'])

In [35]:
recommendations = recommendations.sort_values(by='predicted_rating',ascending=False)

In [36]:
movie_info = df[['movieId', 'title', 'genres']].drop_duplicates()

recommendations = recommendations.merge(movie_info,on='movieId')

In [37]:
recommendations[['title', 'genres', 'predicted_rating']].head(10)

,title,genres,predicted_rating
0,"Color of Paradise, The (Rang-e khoda) (1999)",Drama,4.720549
1,"Lion in Winter, The (1968)",Drama,4.696932
2,Steamboat Willie (1928),Animation|Children|Comedy|Musical,4.623403
3,Turbo: A Power Rangers Movie (1997),Action|Adventure|Children,4.609918
4,Abbott and Costello Meet Frankenstein (1948),Comedy|Horror,4.542552
5,Incredibly True Adventure of Two Girls in Love...,Comedy|Romance,4.538765
6,Love Affair (1994),Drama|Romance,4.513090
7,Soft Toilet Seats (1999),Comedy,4.508519
8,Duck Soup (1933),Comedy|Musical|War,4.486115
9,Caligula (1979),Drama,4.482996


In [38]:
model = SVD()

model.fit(trainset)

predictions = model.test(testset)

rmse(predictions)

RMSE: 1.0296


np.float64(1.029575274253954)

In [41]:
df

,userId,movieId,rating,title,genres
0,1488844,1,3,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,822109,1,5,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,885013,1,4,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,30878,1,4,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,823519,1,3,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
...,...,...,...,...,...
23813498,2591364,4499,2,Dirty Rotten Scoundrels (1988),Comedy
23813499,1791000,4499,2,Dirty Rotten Scoundrels (1988),Comedy
23813500,512536,4499,5,Dirty Rotten Scoundrels (1988),Comedy
23813501,988963,4499,3,Dirty Rotten Scoundrels (1988),Comedy


#**3. Find what Genre Movies have received the best and worst ratings based on User Rating.**

In [44]:
genre_rating = df.groupby('genres')['rating'].mean()

print("Best Rated Genre:")
print(genre_rating.idxmax())
print("Average Rating:", genre_rating.max())

print()

print("Worst Rated Genre:")
print(genre_rating.idxmin())
print("Average Rating:", genre_rating.min())

Best Rated Genre:
Action|Crime|Mystery|Thriller
Average Rating: 4.47076146403029

Worst Rated Genre:
Adventure|Comedy|Romance|War
Average Rating: 2.053030303030303
